# Глава 2. Эмбединги и первые нейросетевые модели
В области обработки естественного языка компьютеры уже умели немало: искать документы по ключевым словам, считать частоты, предсказывать следующее слово по статистике коротких цепочек. Но принципиальной проблемой оставалось то, что для машины слова «кошка» и «кот» были так же далеки друг от друга, как «кошка» и «трактор»: две строки либо совпадают, либо нет, не существует промежуточного состояния. Потребность создать такое представления, слово превратилось из просто "символьной метки" в точку непрерывного векторного пространства смыслов, в котором геометрическая близость отражает близость смысловую.

В этой главе рассмотрим модель Бенжио первую нейросетевую модель языка. Обсудтим ставшую революционной модель построения статических эмбедингов Word2Vec, а также ее более поздние модификации GloVe и FastText.

## Модель Бенжио
До 2000 года методы решения задачи языкового моделирования (предсказать следующее слово по предыдущим) в основном ограничивались n-граммными моделями. Это был рабочий интсрумент для базовыхх задач (например, автозаполнение), но в задачах требующих более глубкого понимания текста их не хватало. У n-граммных моделей была фундаментальная беда - то самое «проклятие размерности».

Во-первых, число возможных комбинаций из $n$ слов астрономическое $({|V|}^n)$. Например, при использовании скромных 10000 слов, словарь триграммов предполагает триллион возможных комбинаций, большинство из которых ни разу не встречается в обучающих данных. Соотвественно если где-то в тексте встречается такая комбинация, она оценивается моделью, как имеющая нулевую вероятностью и ее приходится искусственно коректиовать инженерными приемами.

Во-вторых, слова в таких моделях - дискретные символьные наборы, лишенные семантики. Например, «кот» и «собака» для них различаются ровно так же, как «кот» и «пылесос», это просто разные индексы в словаре. Из-за этой дисккретности модель не имеет способности нормально обобщать входные данные, навык совершенно необходимый для любого более или менее серьезного моделирования.

Обе проблемы были бы решены, если бы в эти дискретные представления мы как-то добавили "непрерывность". В 2003 году [(Benjio et al)](https://jmlr.org/papers/volume3/tmp/bengio03a.pdf) предложили сопоставлять каждому слову из словаря некоторый обучаемый вещественный вектор (порядка 30–100 измерений), который выполнял роль его непрерывного представления. Таким образом, представления слов стали описывать "карту смыслов", где каждое слово занимает свое место, а схожие слова находятся рядом.

Вероятность же продолжения вместо вычисленной статистики стала моделироваться несложной двуслойной нейронной сетью. Фактически это был первый пример использования обучаемых эмбедингов, стандарт описания текстовых данных, использующийся до сих пор. Хотя само слово "эмбединг" популяризровали существенно позже после выхода модели Word2Vec (о ней поговорим ниже).

Что дала непрерыввность представления - она включила обобщаемость: если в ходе обучения «кот» и «собака» получают близкие векторы, то модель, видевшая «кот сидел на полу», автоматически назначит разумную вероятность фразе «собака сидела на полу», даже если её в данных не было. Похожие слова → близкие векторы → переносимое знание. Вот так непрерывное пространство признаков лечит проклятие размерности.

<img src="img/benjio.png" width=400>

Архитектурно это feed-forward нейросеть: эмбединги $n-1$ предыдущих слов $C(w_{t-n+1}), \dots, C(w_{t-1})$ конкатенируются в единый вектор $x$, проходят через скрытый слой с нелинейностью, а на выходе softmax по всему словарю $V$ даёт распределение вероятностей следующего слова:

$$\hat{P}(w_t \mid w_{t-n+1}, \dots, w_{t-1}) = \frac{e^{y_{w_t}}}{\sum_{i \in V} e^{y_i}}, \qquad y = b + W x + U \tanh(d + H x).$$

Здесь $y_i$ — ненормированная оценка (logit) для $i$-го слова словаря, а матрицы $H, U, W$ и векторы смещений $b, d$ — обучаемые параметры сети. Обучается всё это максимизацией логарифма правдоподобия обучающего текста (с регуляризацией):

$$L = \frac{1}{T}\sum_{t} \log \hat{P}(w_t \mid w_{t-n+1}, \dots, w_{t-1})$$

При всех плюсах у модели оставалась пара фундаментальных узких мест. Во-первых, модель использует контекст фиксированной и как правило небольшой длины (всего несколько десятков токенов)<br>Естественно, это сильно ограничивает возможности модели, поскольку она может держать контекст максимум на уровне 1-2 предложений средней длины. Для обхода этого ограничения нужно либо возвращаться к bag-of-words моделям, где текст описывается глобально, либо использовать рекурсивные сети, о которых речь пойдет в следующей главе.

Во-первых, вычисление знаменателя в формуле softmax крайне дорогостоящая операция из-за того, что необходимо каждый раз считать сумму по всему словарю, который в совеременных моделях может содержать десятки и даже сотни тысяч слов:
$$\text{softmax}(y_i) = \frac{e^{\text{y}_i}}{e^{\text{y}_1} + e^{\text{y}_2} + ... + e^{\text{y}_{|V|}}}$$

В первой версии модели авторы боролись с этим эффектом просто распараллеливанием подсчета по нескольким процессорам, но в более поздних модификациях стали искать более умный подход и заменили softmax на приближенное вычисление. Ниже перечислим несколько инженерных приемов, со временем ставших классическими для ускорения вычисления софтмакса:
- Иерархический софтмакс (hierarchical softmax), подход, предложенный самими авторами во второй версии своей модели [(Bengio et al, 2005)](https://proceedings.mlr.press/r5/morin05a.html)
- Сэмплирование по важности (importance sampling)
- Выборочный софтмакс (Sampled Softmax)

### Практические приемы оптимизации

#### Иерархический softmax

Вместо того, чтобы вычислять вероятности для всех слов выходного слоя (что требует O(|V|) операций), метод строит бинарное дерево, где листьями являются слова, а внутренними узлами — бинарные классификаторы. Вероятность конкретного слова вычисляется как произведение вероятностей прохождения по пути от корня к соответствующему листу, где на каждом узле принимается решение (левая или правая ветвь) с использованием сигмоид-функции. Это снижает вычислительную сложность с O(|V|) до O(log |V|) (средняя глубина дерева), что позволяет обучать модели на миллионных словарях, однако на практике качество такой аппроксимации может немного уступать полному softmax, а её производительность сильно зависит от структуры дерева (часто используется дерево Хаффмана для ускорения на частых словах), и в современных архитектурах она всё чаще уступает место другим методам вроде негативного сэмплирования, оставаясь при этом фундаментальным алгоритмом в истории NLP.

Вместо проекции в словарь обучается бинарное дерево, по которому восстанавливается вероятность токена при заданном конексте h. 

Идея иерархического softmax: параметризуем не слова, а внтуренние ноды дерева поиска (листья - слова). Каждое такое представление внутренней ноды перенаправляет $h$ влево или вправо. Таким образом для подсчета вероятности $P(w|h)$ нужно сделать $log(|V|)$ шагов. В модели Bengio дерево строили по WordStat

#### Importance Sampling
Сэмплирование по важности (importance sampling) - это классический прием из статистики, использующийся для сэмплирования наблюдений из сложного распредлеения, и известный еще с 1940 годов. Он основан на простой идее, что когда нужно сэмплировать случайный сэмпл из сложного распределдения P можно приближенно заменить на сэмплирование из простого распржедения Q, если добавить корректирующую поправку.

В контексте вычисления softmax в языковой модели, используя тот принцип, можно заменить полный расчет знаменателя на статистику, посчитанную на небольшой выборке (скажем, 100-200 штук). Примеры сэмплируют из более простого распределения, например, равномерного - просто случайно выбираем токены. Разницу между распределениями надо чем то компенсировать, поэтому каждое наблюдение взвешивают: если пример часто встречается в Q, его берут с большим весом, если редко, то с меньшим весом.

Если обозначим за S множество случайно выбранных токенов, тогда вероятноть токена оценивается как:

$$\text{softmax}(y_i) = \frac{e^{\text{y}_i}}{\sum_{j \in V} e^{\text{y}_j}} \approx \frac{e^{\text{y}_i}}{\sum_{j \in S} e^{\text{y}_j} / q(j)}$$

Этот прием используется только на обучении, на генерации все же нужно перербрать все варианты. Да, оценка несмещнная (на большом количестве расчетов оценивает вероятность достаточно точно), но диспесрия остается большой и ей нельзя принебречь. Кроме того, порядок токенов важнее абсолютных вероятностей.

Разница в скорости вычислений обычно несколько сотен раз.

#### Sampled Softmax
В IS оценка вероятности смещенная, несмещенным будет градиент. Альтернативно можно заменить саму функцию потерь, это называется Sampled Softmax.



## Модель Word2Vec
В 2013 году [(Mikolov et al.)](https://arxiv.org/abs/1301.3781) с коллегами из Google задался прагматичным вопросом: если нужны только векторы слов, зачем обучать полноценную языковую модель? Так появилось переосмсыление модели Бенжио, модель **Word2Vec** . Модель была намеренно упрощена: из неё выброшен дорогой скрытый нелинейный слой и осталась только линейная проекция. С этого момента термин эмбединг стал повсеместным и его начали массово использовать

Контекстом будем называть слова соседние относительно центрального слова. Сформулировал два варианта постановки:
__CBOW__ (continuous bag-of-words), в рамках которой модель учится предсказывать центральное слово по окружающим его слева и справа словам и __skip-gram__, в рамках которого модель учится по центральному слову предсказывать окружающие. Важно что результат одинаковый, мы получаем скрытые представления

Оба подхода сопоставимы по качеству. Skip-gram обычно даёт лучшие представления для редких слов, но CBOW работает быстрее.

Обратите внимание, что обе архитектуры используют контекст с обеих сторон от слова, а не только предшествующие слова, как языковая модель Бенжио

Формально skip-gram максимизирует среднюю логарифмическую вероятность контекстных слов при данном центральном слове $w_t$ в окне радиуса $c$:

$$\frac{1}{T}\sum_{t=1}^{T} \sum_{-c \le j \le c,\; j \ne 0} \log p(w_{t+j} \mid w_t),$$

где базовая (softmax) параметризация использует два набора векторов — «входной» $\mathbf{v}_w$ для центрального слова и «выходной» $\mathbf{u}_w$ для контекстного:

$$p(w_O \mid w_I) = \frac{\exp(\mathbf{u}_{w_O}^{\top}\mathbf{v}_{w_I})}{\sum_{w \in V}\exp(\mathbf{u}_{w}^{\top}\mathbf{v}_{w_I})}$$

Проблема вычисления знаменателя в Softmax никуда не делась. Mikolov предложил два пути решения:<Br>
а) уже знакомый иерархический softmax - параметризуем внутренние ноды дерева поиска. Здесбь отличие в том, что дерево предстваляет собой дерево Хаффмана - структуру, учитывающкю частоту слова<br>
б) в более поздней версии использовался negative sampling:  вместо нормировки по всему словарю модель учится отличать реальную пару «слово — контекст» от $k$ случайно подобранных «отрицательных» примеров $w_i \sim P_n(w)$:
$$\log \sigma(\mathbf{u}_{w_O}^{\top}\mathbf{v}_{w_I}) + \sum_{i=1}^{k} \mathbb{E}_{w_i \sim P_n(w)}\big[\log \sigma(-\mathbf{u}_{w_i}^{\top}\mathbf{v}_{w_I})\big],$$
где $\sigma(x) = 1/(1+e^{-x})$. Именно эти приёмы позволили обучать качественные векторы на миллиардах слов меньше чем за день.

Особенную популярность метод получил благодаря удивительному наблюдению - оказалось, что полученное пространство обладает линейной структурой: семантические и синтаксические отношения выражаются постоянными векторными сдвигами. Отсюда знаменитая арифметика аналогий — $\mathbf{v}_{\text{король}} - \mathbf{v}_{\text{мужчина}} + \mathbf{v}_{\text{женщина}} \approx \mathbf{v}_{\text{королева}}$

## Модель GloVe
К 2014 году в области существовало две группы подходов для текстов: первая - это методы тип латентно-семантиеского анализа, работающие с глобальной статистикой (co-occurence matrix) и параметрические методы типа Word2Vec, обучающиеся на локальном контексте. Команда [(Pennington et al., 2014)](https://aclanthology.org/D14-1162/) из Стенфорда попыталась изобрести гибридный подход, сочетающий сильные стороны каждой из двух групп и назвали его **GloVe** (Global Vectors).

Строится глобальная матрица совместной встречаемости $X$, где $X_{ij}$ — сколько раз слово $j$ встречается в контексте слова $i$ по всему корпусу. Ключевое наблюдение авторов: осмысленную информацию несут не сами по себе совместные встречаемости, а их отношения. Отношение вероятностей $P_{ik}/P_{jk}$, где $P_{ik} = X_{ik}/\sum_l X_{il}$ — вероятность встретить пробное слово $k$ рядом со словом $i$, хорошо разделяет релевантные и нерелевантные ассоциации: оно велико, если $k$ ближе к $i$, чем к $j$, и мало в обратном случае. Из этого наблюдения выводится взвешенная задача наименьших квадратов, в которой скалярное произведение векторов приближает логарифм числа совместных встречаемостей:

$$J = \sum_{i,j=1}^{V} f(X_{ij})\,\big(\mathbf{w}_i^{\top}\tilde{\mathbf{w}}_j + b_i + \tilde{b}_j - \log X_{ij}\big)^2.$$

Здесь $\mathbf{w}_i$ и $\tilde{\mathbf{w}}_j$ — векторы слова и контекстного слова, $b_i, \tilde{b}_j$ — смещения, а весовая функция $f$ гасит вклад как слишком редких, так и чересчур частых пар:

$$f(x) = \begin{cases} (x/x_{\max})^{\alpha}, & x < x_{\max} \\ 1, & x \ge x_{\max} \end{cases}$$

с типичными $x_{\max} = 100$ и $\alpha = 3/4$.

Расчёт оправдался. GloVe обучается не проходом по тексту скользящим окном, а по уже собранной матрице статистик, что делает его эффективным и хорошо распараллеливаемым. По качеству на задачах аналогий и сходства слов он оказался сопоставим с Word2Vec — и на несколько лет пара Word2Vec / GloVe стала стандартным набором предобученных эмбеддингов, которыми инициализировали модели в огромном числе прикладных NLP-задач.

## Модель FastText
В 2016 году тот же Томаш Миколов работал уже в Facebook AI Research и вместе с коллегами они доработали подход к Word2Vec и GloVe. У стандартной модели эмбедингов было два главных ограничения. Во-первых она полностью игнорирует морфологию: слова «бежать», «бежит», «убежал» кодируются разными векторами, хотя очевидно родственны. Это особенно критично для морфологически богатых языков, таких как например русский. Во-вторых, модели беспомощны перед ранее не встречавшимися словами (out-of-vocabulary): для этих слов свой эмбединг просто не предусмотрен (не то чтобы это было частой проблемой, но иногда встречающася).

Чтобы побороть эти ограничения, авторы перевели модель с уровня слов на уровень отдельных символов и назвали ее **FastText** [(Bojanowski et al., 2016)](https://arxiv.org/abs/1607.04606), Модель обучения оставили ту же skip-gram, но теперь каждое слово описывается как набор *символьных* n-грамм - подстрок, сотоящих из трёх–шести символов. 

Каждой такой n-грамме $g$ сопоставляется свой вектор $\mathbf{z}_g$. Оценка совместимости анализируемого слова $w$ (с множеством его n-грамм $\mathcal{G}_w$) и контекстного слова $c$ считается не как одно скалярное произведение, а как сумма по n-граммам:

$$s(w, c) = \sum_{g \in \mathcal{G}_w} \mathbf{z}_g^{\top}\mathbf{v}_c$$

Благодаря такому способу разибения модель стала учитывать морфологию: слова с общими корнями и аффиксами разделяют символьные n-граммы, а значит, автоматически получают близкие векторы. Кроме того, для любого нового слова, не встречавшегося при обучении, вектор можно собрать из его символьных n-грамм и проблема out-of-vocabulary фактически снимается. При этом метод остаётся быстрым и обучается на больших корпусах так же эффективно, как Word2Vec.

FastText во многом стал стандартом *статических* эмбеддингов. Дальнейшее равитие идеи дистрибутивных векторов уперлось в проблему независимости этих представлений от контекста. И такая независимость от контекста - самое большое ограничение, для преодоления которого стали разрабатывать целую линейку более современных подходов, сначала на базе рекурсивных моделей (подробнее в главе 3), а потом Трансформерных (подробнее в главе 4).

Важно отметить, что FastText это не только модель эмбедингов, но и разработанная библиотека с методами классификации. Если в течение нескольких лет после своего выхода FastText был популярен скорее как источник эмбедингов, то в дальнейшем он стал стандартом именно в области классикации и предобатки больших корпусов текстов. Библиотеку до сих пор активно используют, когда нужно отфильтровать текст (например, см. процесс обучения модели DeepSeek-Math) или определить его язык (модель lid.176 отраслевой стандарт).

